# Retail Sales Performance: Cleaning and Exploration
**Business question:** which stores and categories miss target, and why? (May to Oct 2026, 4 stores x 3 categories)

Steps 3 (Clean) and 4 (Explore) of the 7-step workflow. Steps 5-6 (KPIs, dashboard) are in the Excel file.

In [1]:
import pandas as pd
df = pd.read_csv('store_sales_raw.csv')
print(df.shape)
df.info()

(74, 8)
<class 'pandas.DataFrame'>
RangeIndex: 74 entries, 0 to 73
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Month     74 non-null     str    
 1   Store     74 non-null     str    
 2   Region    74 non-null     str    
 3   Category  74 non-null     str    
 4   Visitors  72 non-null     float64
 5   Orders    74 non-null     int64  
 6   Revenue   74 non-null     str    
 7   Target    74 non-null     int64  
dtypes: float64(1), int64(2), str(5)
memory usage: 4.8 KB


## Step 3: Clean
### 3.1 Data quality checks

In [2]:
print(df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())
print(df['Store'].unique())
print(df['Category'].unique())

Month       0
Store       0
Region      0
Category    0
Visitors    2
Orders      0
Revenue     0
Target      0
dtype: int64
Duplicate rows: 2
<StringArray>
['Pune', 'Mumbai', 'pune ', 'Nashik', 'Nagpur', 'NAGPUR']
Length: 6, dtype: str
<StringArray>
['Apparel', 'Footwear', 'Accessories', 'Foot wear']
Length: 4, dtype: str


### 3.2 Remove exact duplicate rows (would double-count revenue)

In [3]:
df = df.drop_duplicates()
df.shape

(72, 8)

### 3.3 Fix store name spelling

In [4]:
df['Store'] = df['Store'].str.strip().str.replace('pune', 'Pune').str.replace('NAGPUR', 'Nagpur')
df.groupby('Store').size()

Store
Mumbai    16
Nagpur    19
Nashik    18
Pune      19
dtype: int64

### 3.4 Grain check: every store must have 18 rows (3 categories x 6 months)
Mumbai has 16, Pune and Nagpur 19. Two rows carry the wrong store name.

In [5]:
df[df.duplicated(subset=['Month', 'Store', 'Category'], keep=False)]

,Month,Store,Region,Category,Visitors,Orders,Revenue,Target
2,May-2026,Pune,West,Accessories,1458.0,44,20143,24000
5,May-2026,Pune,West,Accessories,2267.0,58,32722,34000
40,Aug-2026,Nagpur,West,Footwear,3839.0,94,105620,111000
46,Aug-2026,Nagpur,Central,Footwear,2797.0,65,91687,97000


Evidence: row 5 has Accessories target 34,000 (Mumbai's, Pune's is 24,000) and row 40 has Region = West with Footwear target 1,11,000 (Mumbai's; Nagpur is Central with 97,000). Both rows belong to **Mumbai**.

In [6]:
df.loc[5, 'Store'] = 'Mumbai'
df.loc[40, ['Store', 'Region']] = ['Mumbai', 'West']
df.groupby('Store').size()

Store
Mumbai    18
Nagpur    18
Nashik    18
Pune      18
dtype: int64

### 3.5 Fix category label
'Foot wear' in Jul Nagpur has target 1,25,000, which is Nagpur's **Apparel** target, and Nagpur already has a Jul Footwear row. So it is Apparel.

In [7]:
df.loc[33, 'Category'] = 'Apparel'
df.groupby(['Store', 'Category']).size().unique()  # should be [6]

array([6])

### 3.6 Revenue stored as text ('Rs 98,927') -> number

In [8]:
df['Revenue'] = pd.to_numeric(df['Revenue'].astype(str).str.replace('Rs', '', regex=False).str.replace(',', '', regex=False).str.strip())

### 3.7 Missing Visitors (2 rows) -> median of the same Store + Category
An overall mean (3,032) would triple Nashik Accessories (~1,000) and halve Mumbai Apparel (~6,500).

In [9]:
group_median = df.groupby(['Store', 'Category'])['Visitors'].transform('median')
df['Visitors'] = df['Visitors'].fillna(group_median).astype(int)
df.loc[[20, 63]]

,Month,Store,Region,Category,Visitors,Orders,Revenue,Target
20,Jun-2026,Nashik,West,Accessories,1030,37,15843,15000
63,Oct-2026,Mumbai,West,Apparel,6271,158,138316,142000


### 3.8 Month text -> date

In [10]:
df['Month'] = pd.to_datetime(df['Month'], format='%b-%Y')
print(df.shape)
print(df.dtypes)
print('Total revenue:', df['Revenue'].sum())
df.to_csv('store_sales_clean.csv', index=False)

(72, 8)
Month       datetime64[us]
Store                  str
Region                 str
Category               str
Visitors             int64
Orders               int64
Revenue              int64
Target               int64
dtype: object
Total revenue: 5231549


## Step 4: Explore

In [11]:
store = df.groupby('Store').agg(revenue=('Revenue', 'sum'), target=('Target', 'sum'))
store['variance_pct'] = ((store['revenue'] - store['target']) / store['target'] * 100).round(1)
store

,revenue,target,variance_pct
Store,,,
Mumbai,1712496,1722000,-0.6
Nagpur,1495697,1512000,-1.1
Nashik,790598,774000,2.1
Pune,1232758,1224000,0.7


In [12]:
month = df.groupby('Month').agg(revenue=('Revenue', 'sum'), target=('Target', 'sum'))
month['variance_pct'] = ((month['revenue'] - month['target']) / month['target'] * 100).round(1)
month

,revenue,target,variance_pct
Month,,,
2026-05-01,787365,872000,-9.7
2026-06-01,884701,872000,1.5
2026-07-01,838110,872000,-3.9
2026-08-01,902030,872000,3.4
2026-09-01,903996,872000,3.7
2026-10-01,915347,872000,5.0


In [13]:
hit = (df['Revenue'] >= df['Target']).sum()
print(f'{hit} of {len(df)} store-category-months hit target ({hit/len(df):.0%})')

31 of 72 store-category-months hit target (43%)


**Observations**
- Nagpur was the weakest store: Rs 14.96 L vs target Rs 15.12 L (-1.1%). Mumbai also missed (-0.6%). Nashik (+2.1%) and Pune (+0.7%) beat target.
- May was the weakest month: Rs 7.87 L vs Rs 8.72 L (-9.7%). July also missed (-3.9%); Aug to Oct each beat target by 3-5%.
- Only 31 of 72 store-category-months (43%) hit target.